### Dependencies

In [4]:
!pip install -q --upgrade transformers

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 10.2/10.2 MB 89.1 MB/s eta 0:00:00
  Attempting uninstall: transformers
    Found existing installation: transformers 5.0.0
    Uninstalling transformers-5.0.0:
      Successfully uninstalled transformers-5.0.0


In [2]:
!pip install -q rouge-score nltk bert-score requests beautifulsoup4 PyMuPDF

In [5]:
!pip show transformers

Name: transformers
Version: 5.5.4
Summary: Transformers: the model-definition framework for state-of-the-art machine learning models in text, vision, audio, and multimodal models, for both inference and training.
Home-page: https://github.com/huggingface/transformers
Author: The Hugging Face team (past and future) with the help of all our contributors (https://github.com/huggingface/transformers/graphs/contributors)
Author-email: transformers@huggingface.co
License: Apache 2.0 License
Location: /usr/local/lib/python3.12/dist-packages
Requires: huggingface-hub, numpy, packaging, pyyaml, regex, safetensors, tokenizers, tqdm, typer
Required-by: bert-score, peft, sentence-transformers


### Data Collecting (Web Scraping)

#### LOINC

In [2]:
SECTION7_LOINC = "34073-7"
SECTION12_LOINC = "34090-1"

#### Text Retrieval Functions

In [3]:
import re
import requests
from bs4 import BeautifulSoup
from xml.etree import ElementTree
import fitz

def extract_setId(url):
  regMatch = re.search(r'[?&]setid=([a-f0-9]{8}-(?:[a-f0-9]{4}-){3}[a-f0-9]{12})', url)

  if regMatch:
    setId = regMatch.group(1)
    print(f"Extracted ID: {setId}")
    return setId
  return None

def getDailyMedDrugInfo(setId):
  try:
    # API endpoint
    endpoint = f"https://dailymed.nlm.nih.gov/dailymed/services/v2/spls/{setId}.xml"

    response = requests.get(endpoint)
    response.raise_for_status()

    tree = ElementTree.fromstring(response.content)

    section7 = tree.find(".//{urn:hl7-org:v3}code[@code='" f"{SECTION7_LOINC}']/..")
    section12 = tree.find(".//{urn:hl7-org:v3}code[@code='" f"{SECTION12_LOINC}']/..")

    section7Text = None
    section12Text = None

    if section7 is not None:
      section7Text = "".join(section7.itertext()).strip()
    if section12 is not None:
      section12Text = "".join(section12.itertext()).strip()

    return section7Text, section12Text
  except Exception as e:
    return getPdfDrugInfo("https://dailymed.nlm.nih.gov/dailymed/downloadpdffile.cfm?setId="+setId)

def getHtmlDrugInfo(url):
  r = requests.get(url.strip())
  r.raise_for_status()

  soup = BeautifulSoup(r.text, 'html.parser')

  section7 = soup.find('div', attrs={"data-sectionCode": SECTION7_LOINC})
  section12 = soup.find('div', attrs={"data-sectionCode": SECTION12_LOINC})

  section7Text = None
  section12Text = None

  if section7 is not None:
    section7Text = section7.text
  if section12 is not None:
    section12Text = section12.text

  if section7Text is None or section12Text is None:
    return getHtmlFallbackDrugInfo(url)

  return section7Text, section12Text

def getHtmlFallbackDrugInfo(url):
    r = requests.get(url.strip())
    r.raise_for_status()
    soup = BeautifulSoup(r.text, 'html.parser')

    for script_or_style in soup(["script", "style"]):
        script_or_style.decompose()

    full_text = soup.get_text(separator="\n")

    # We remove the \n constraint slightly for HTML but keep the 'longest match' logic
    patterns = {
        "7": r"7\s+DRUG\s+INTERACTIONS(.*?)(?=\n\s*\d+\s+[A-Z]|\Z)",
        "12": r"12\s+CLINICAL\s+PHARMACOLOGY(.*?)(?=\n\s*\d+\s+[A-Z]|\Z)"
    }

    results = {"7": None, "12": None}

    for code, pattern in patterns.items():
        matches = re.findall(pattern, full_text, re.IGNORECASE | re.DOTALL)
        if matches:
            # TOC entries are usually ~10-100 chars. Real sections are > 200.
            # max(key=len) effectively skips the TOC entry.
            best_match = max(matches, key=len).strip()
            if len(best_match) > 80:
                results[code] = best_match

    return results["7"], results["12"]

def getPdfDrugInfo(url):
    response = requests.get(url)
    # Using a list to track all matches so we can pick the one that isn't TOC
    found_sections = {"7": None, "12": None}

    with fitz.open(stream=response.content, filetype="pdf") as doc:
        full_text = ""
        for page in doc:
            full_text += page.get_text("text")

    patterns = {
        # This lookahead ensures that 'DRUG INTERACTIONS' is NOT followed by dots/numbers then a newline (TOC style)
        "7": r"(?:\n7\s+DRUG\s+INTERACTIONS\s*\n)(.*?)(?=\n\s*\d+\s+[A-Z]|\Z)",
        "12": r"(?:\n12\s+CLINICAL\s+PHARMACOLOGY\s*\n)(.*?)(?=\n\s*\d+\s+[A-Z]|\Z)"
    }

    for code, pattern in patterns.items():
        # Using findall because the TOC match usually comes first
        # If the specific header-style regex fails, we find all and take the longest one
        matches = re.findall(pattern, full_text, re.IGNORECASE | re.DOTALL)
        if matches:
            # Sort by length and take the longest one (actual content > TOC entry)
            found_sections[code] = max(matches, key=len).strip()
        else:
            found_sections[code] = None

    return found_sections["7"], found_sections["12"]

#### Load URLs from Excel

In [4]:
import pandas as pd
import io
from google.colab import files

print("Click the button below to upload your Excel file...")
uploaded = files.upload()

df = pd.read_excel(io.BytesIO(list(uploaded.values())[0]))
urls = df['URL'].tolist()
print(f"\nLoaded {len(urls)} URLs successfully\n")

Click the button below to upload your Excel file...


Saving 100 Drug URLs 2026_04_07.xlsx to 100 Drug URLs 2026_04_07.xlsx

Loaded 100 URLs successfully



In [5]:
import requests

def getDrugText(url):
    try:
        response_head = requests.head(url, allow_redirects=True, timeout=10)
        content_type = response_head.headers.get('Content-Type', '').lower()

        if 'application/pdf' in content_type or url.strip().lower().endswith(".pdf"):
            print(f"Detected PDF")
            return getPdfDrugInfo(url)
        elif 'text/html' in content_type:
            print(f"Detected HTML")
            return getHtmlDrugInfo(url)
        else:
            print(f"[!] Unknown content type: {content_type}")

    except Exception as e:
        print(f"[!] Error accessing URL: {e}")

In [6]:
import io
import time

master_drug_data = []
failures = []

def extract_drug_text(url):
  try:
    setId = extract_setId(url)
    if setId is not None:
      section7Text, section12Text = getDailyMedDrugInfo(setId)
      if section7Text is None or section12Text is None:
        section7Text, section12Text = getDrugText(url)
    else:
      section7Text, section12Text = getDrugText(url)

    if section7Text is None and section12Text is None:
      return "NA", "NA", False

    return section7Text, section12Text, True
  except Exception as e:
    return "NA", str(e), False

print("Scraping URLs...\n")

def add_err(iter, err):
  failures.append({"index": iter+1, "url": url, "error": err})
  print(f"  [{iter+1}/{len(urls)}] FAILED — {url[:70]}, failure: {len(failures)}, error: {err}")

for i, url in enumerate(urls):
  print(f"Processing {url}")

  try:
    sec7, sec12, success = extract_drug_text(url)
    if success and (
      (sec7 is not None and len(sec7) > 80) or
      (sec12 is not None and len(sec12) > 80)
    ):
      master_drug_data.append({"index": i+1, "url": url, "section7": sec7, "section12": sec12})
      print(f"  [{i+1}/{len(urls)}] OK — {url[:70]}")
    else:
      add_err(i, sec12)
  except Exception as e:
    add_err(i, str(e))

  print("----------------------------------------")
  time.sleep(0.15)

print(f"\n✓ Done — {len(master_drug_data)} succeeded, {len(failures)} failed.")

Scraping URLs...

Processing https://labeling.pfizer.com/ShowLabeling.aspx?id=16474 
Detected PDF
  [1/100] OK — https://labeling.pfizer.com/ShowLabeling.aspx?id=16474 
----------------------------------------
Processing https://dailymed.nlm.nih.gov/dailymed/fda/fdaDrugXsl.cfm?setid=ca73b519-015a-436d-aa3c-af53492825a1&type=display
Extracted ID: ca73b519-015a-436d-aa3c-af53492825a1
  [2/100] OK — https://dailymed.nlm.nih.gov/dailymed/fda/fdaDrugXsl.cfm?setid=ca73b51
----------------------------------------
Processing https://uspl.lilly.com/verzenio/verzenio.html#pi
Detected HTML
  [3/100] OK — https://uspl.lilly.com/verzenio/verzenio.html#pi
----------------------------------------
Processing https://www.janssenlabels.com/package-insert/product-monograph/prescribing-information/ZYTIGA-pi.pdf
Detected PDF
  [4/100] OK — https://www.janssenlabels.com/package-insert/product-monograph/prescri
----------------------------------------
Processing https://dailymed.nlm.nih.gov/dailymed/fda/fdaD

#### Save Collected Data to CSV

In [7]:
import csv

keys = master_drug_data[0].keys()

with open('collected_drug_data.csv', 'w', newline='') as output_file:
    dict_writer = csv.DictWriter(output_file, keys)
    dict_writer.writeheader()
    dict_writer.writerows(master_drug_data)

print("Data saved to 'collected_drug_data.csv'")

Data saved to 'collected_drug_data.csv'


In [8]:
import csv

keys = failures[0].keys()

with open('failures_drug_data.csv', 'w', newline='') as output_file:
    dict_writer = csv.DictWriter(output_file, keys)
    dict_writer.writeheader()
    dict_writer.writerows(failures)

print("Failures saved to 'failures_drug_data.csv'")

Failures saved to 'failures_drug_data.csv'


#### Optionally Provide a CSV Drug Data File

In [7]:
from google.colab import files
import csv
import sys

print("Click the button below to upload the CSV file...")
uploaded = files.upload()

for fn in uploaded.keys():
  print('User uploaded file "{name}" with length {length} bytes'.format(
      name=fn, length=len(uploaded[fn])))

filename = next(iter(uploaded))

csv.field_size_limit(sys.maxsize)

master_drug_data = []
with open(filename, newline='') as csvfile:
    reader = csv.DictReader(csvfile)
    for row in reader:
        master_drug_data.append(row)

print(master_drug_data)

Click the button below to upload the CSV file...


Saving collected_drug_data(3).csv to collected_drug_data(3).csv
User uploaded file "collected_drug_data(3).csv" with length 737435 bytes
[{'index': '1', 'url': 'https://labeling.pfizer.com/ShowLabeling.aspx?id=16474 ', 'section7': '7.1 \nPotential for PAXLOVID to Affect Other Drugs \n \nPAXLOVID (nirmatrelvir co-packaged with ritonavir) is a strong inhibitor of CYP3A, and an inhibitor of \nCYP2D6, P-gp and OATP1B1. Co-administration of PAXLOVID with drugs that are primarily \nmetabolized by CYP3A and CYP2D6 or are transported by P-gp or OATP1B1 may result in increased \nplasma concentrations of such drugs and increase the risk of adverse events. Co-administration of \nPAXLOVID with drugs highly dependent on CYP3A for clearance and for which elevated plasma \nconcentrations are associated with serious and/or life-threatening events is contraindicated [see \nContraindications (4) and Drug Interactions (7.3) Table 2]. Co-administration with other CYP3A \nsubstrates may require a dose adju

In [8]:
def find_best_entry(data_list):
    """
    Returns the complete dictionary that contains the longest
    text in either section 7 or section 12.
    """

    def scoring_criteria(d):
        # We handle None or missing keys by defaulting to an empty string
        s7_content = d.get('section7') or ""
        s12_content = d.get('section12') or ""

        # The 'score' is the length of the longest string found in this dict
        return max(len(s7_content), len(s12_content))

    if not data_list:
        print("Debug: List is empty.")
        return None

    # max() returns the FULL dictionary 'd' from the list
    best_dict = max(data_list, key=scoring_criteria)

    # --- Debug Summary ---
    print(f"--- Winner Stats ---")
    print(f"Section 7 Length:  {len(best_dict.get('section7') or '')}")
    print(f"Section 12 Length: {len(best_dict.get('section12') or '')}")
    print(f"Full Dict Keys:    {list(best_dict.keys())}")

    return best_dict

find_best_entry(master_drug_data)["url"]
find_best_entry(master_drug_data)["section12"]

--- Winner Stats ---
Section 7 Length:  13291
Section 12 Length: 32709
Full Dict Keys:    ['index', 'url', 'section7', 'section12']
--- Winner Stats ---
Section 7 Length:  13291
Section 12 Length: 32709
Full Dict Keys:    ['index', 'url', 'section7', 'section12']


'12\tCLINICAL PHARMACOLOGY\n               \n               \n                  \n                     \n                     \n                     12.1\tMechanism of Action\n                     \n                        Aprepitant is a selective high-affinity antagonist of human substance P/neurokinin 1 (NK1) receptors. Aprepitant has little or no affinity for serotonin (5-HT3), dopamine, and corticosteroid receptors, the targets of existing therapies for chemotherapy-induced nausea and vomiting (CINV).\n                        Aprepitant has been shown in animal models to inhibit emesis induced by cytotoxic chemotherapeutic agents, such as cisplatin, via central actions. Animal and human Positron Emission Tomography (PET) studies with aprepitant have shown that it crosses the blood brain barrier and occupies brain NK1 receptors. Animal and human studies show that aprepitant augments the antiemetic activity of the 5-HT3-receptor antagonist ondansetron and the corticosteroid dexameth

### Summary Instructions

In [9]:
INSTRUCTIONS = (
    "### TASK: MEDICAL SUMMARY ###\n"
    "Generate a high-fidelity abstractive summary of the medical text provided below. "
    "Follow these constraints strictly:\n"
    "1. DO NOT copy sentences; rewrite information into professional, concise clinical language.\n"
    "2. COVER exactly five domains: Indications, Dosing/Administration, Warnings, Side Effects, and Drug Interactions.\n"
    "3. IF DATA IS MISSING for a domain in the provided text, write 'Information not found in source.'\n"
    "4. NO CHATTER: Respond only with the summary. Skip all introductory or meta-commentary.\n\n"
    "--- SOURCE TEXT TO SUMMARIZE ---\n"
)

### LongT5 Inference

#### Remove MedGemma From Memory

In [4]:
import gc
import torch

try:
    del model_mg
    del pipe_mg
    gc.collect()
    torch.cuda.empty_cache()
    print(f"GPU cleared. Free: {torch.cuda.mem_get_info()[0]/1e9:.2f} GB")
except:
    print("No MedGemma in memory, continuing...")

No MedGemma in memory, continuing...


#### Iterate Over Drug Data

In [10]:
import torch
import time
import warnings
import textwrap
from transformers import AutoTokenizer, LongT5ForConditionalGeneration

print("\nLoading Long-T5...")
tokenizer_lt5 = AutoTokenizer.from_pretrained("google/long-t5-tglobal-base")
model_lt5 = LongT5ForConditionalGeneration.from_pretrained(
    "google/long-t5-tglobal-base",
    torch_dtype=torch.float16
).to("cuda")
print("Long-T5 loaded!\n")

def summarize_long_t5(text, max_words=150):
    prompt = INSTRUCTIONS + text
    inputs = tokenizer_lt5(
        prompt,
        return_tensors="pt",
        truncation=True,
        max_length=16384
    ).to(model_lt5.device)
    with warnings.catch_warnings():
        warnings.simplefilter("ignore")
        output_ids = model_lt5.generate(
            inputs["input_ids"],
            min_new_tokens=80,
            max_new_tokens=250,
            num_beams=4,
            length_penalty=2.0,
            no_repeat_ngram_size=4,
            early_stopping=False
        )
    return tokenizer_lt5.decode(output_ids[0], skip_special_tokens=True)

print("Running Long-T5 on all drugs (this will take a while, do not interrupt)...")
lt5_results = []
lt5_times   = []

num_texts = len(master_drug_data) * 2

for i, data in enumerate(master_drug_data):
    for j, text in enumerate([data["section7"], data["section12"]]):
      if text is None:
        continue
      start = time.time()
      output = summarize_long_t5(text)
      elapsed = round(time.time() - start, 2)
      lt5_results.append(output)
      lt5_times.append(elapsed)
      master_drug_data[i][f"LongT5_Summary_{j}"] = output
      master_drug_data[i][f"LongT5_Time_{j}"] = elapsed

      if (i + 1 + j) % 5 == 0 or (i + 1 + j) == num_texts:
          print(f"  Progress: {i+1+j}/{num_texts} done...")

lt5_avg_time = round(sum(lt5_times)/len(lt5_times),2)

print(f"\n✓ LongT5 done. Avg time: {lt5_avg_time}s")


Loading Long-T5...


config.json:   0%|          | 0.00/851 [00:00<?, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

pytorch_model.bin:   0%|          | 0.00/990M [00:00<?, ?B/s]

Loading weights:   0%|          | 0/297 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/147 [00:00<?, ?B/s]

model.safetensors:   0%|          | 0.00/990M [00:00<?, ?B/s]

Long-T5 loaded!

Running Long-T5 on all drugs (this will take a while, do not interrupt)...
  Progress: 5/146 done...
  Progress: 5/146 done...
  Progress: 10/146 done...
  Progress: 10/146 done...
  Progress: 15/146 done...
  Progress: 15/146 done...
  Progress: 20/146 done...
  Progress: 20/146 done...
  Progress: 25/146 done...
  Progress: 25/146 done...
  Progress: 30/146 done...
  Progress: 30/146 done...
  Progress: 35/146 done...
  Progress: 35/146 done...
  Progress: 40/146 done...
  Progress: 40/146 done...
  Progress: 45/146 done...
  Progress: 45/146 done...
  Progress: 50/146 done...
  Progress: 50/146 done...
  Progress: 55/146 done...
  Progress: 55/146 done...
  Progress: 60/146 done...
  Progress: 60/146 done...
  Progress: 65/146 done...
  Progress: 65/146 done...
  Progress: 70/146 done...
  Progress: 70/146 done...

✓ LongT5 done. Avg time: 5.45s


#### Save Results to CSV

In [11]:
import csv

keys = master_drug_data[0].keys()

with open('drug_data_LongT5_results.csv', 'w', newline='') as output_file:
    dict_writer = csv.DictWriter(output_file, keys)
    dict_writer.writeheader()
    dict_writer.writerows(master_drug_data)

print("Data saved to 'drug_data_LongT5_results.csv'")

Data saved to 'drug_data_LongT5_results.csv'


### MedGemma Inference

#### Remove LongT5 From Memory

In [12]:
import os
import gc
import torch
os.environ["PYTORCH_CUDA_ALLOC_CONF"] = "expandable_segments:True"

try:
    del model_lt5
except NameError:
    pass
gc.collect()
torch.cuda.empty_cache()
print(f"GPU cleared. Free: {torch.cuda.mem_get_info()[0]/1e9:.2f} GB")

GPU cleared. Free: 41.84 GB


#### Load MedGemma

In [13]:
from transformers import AutoTokenizer, AutoModelForCausalLM, pipeline
import torch

print("\nLoading MedGemma...")
tokenizer_mg = AutoTokenizer.from_pretrained("google/medgemma-4b-it")
model_mg = AutoModelForCausalLM.from_pretrained(
    "google/medgemma-4b-it",
    dtype=torch.bfloat16,
    device_map="auto"
)
model_mg.generation_config.max_length = None
pipe_mg = pipeline(
    "text-generation",
    model=model_mg,
    tokenizer=tokenizer_mg,
    max_new_tokens=250
)
print("MedGemma loaded!\n")


Loading MedGemma...


config.json: 0.00B [00:00, ?B/s]

tokenizer_config.json: 0.00B [00:00, ?B/s]

tokenizer.json:   0%|          | 0.00/33.4M [00:00<?, ?B/s]

added_tokens.json:   0%|          | 0.00/35.0 [00:00<?, ?B/s]

...

Fetching 2 files:   0%|          | 0/2 [00:00<?, ?it/s]

Loading weights:   0%|          | 0/883 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/156 [00:00<?, ?B/s]

Passing `generation_config` together with generation-related arguments=({'max_new_tokens'}) is deprecated and will be removed in future versions. Please pass either a `generation_config` object OR all generation parameters explicitly, but not both.


MedGemma loaded!



#### Iterate Over Drug Data

In [14]:
from transformers import AutoTokenizer, AutoModelForCausalLM, pipeline
import gc
import torch
import time
import warnings
import textwrap

def chunk_text(text, chunk_size=150):
    words = text.split()
    return [" ".join(words[i:i+chunk_size]) for i in range(0, len(words), chunk_size)]

def medgemma_summarize_chunk(chunk):
    messages = [{"role": "user", "content": (
        "Summarize this medical text in 2-3 sentences capturing key clinical info:\n\n"
        + chunk
    )}]
    formatted = tokenizer_mg.apply_chat_template(
        messages, tokenize=False, add_generation_prompt=True
    )
    with warnings.catch_warnings():
        warnings.simplefilter("ignore")
        output = pipe_mg(formatted, return_full_text=False)
    return output[0]["generated_text"].strip()

def medgemma_base_prompt(text, max_words=150):
    chunks = chunk_text(text, chunk_size=150)
    if len(chunks) == 1:
        return medgemma_summarize_chunk(chunks[0])
    summaries = [medgemma_summarize_chunk(c) for c in chunks]
    combined  = " ".join(summaries)
    messages  = [{"role": "user", "content": (
        f"Combine these into one medical summary paragraph of {max_words} words "
        f"or less. No repeated information:\n\n" + combined
    )}]
    formatted = tokenizer_mg.apply_chat_template(
        messages, tokenize=False, add_generation_prompt=True
    )
    with warnings.catch_warnings():
        warnings.simplefilter("ignore")
        output = pipe_mg(formatted, return_full_text=False)
    return output[0]["generated_text"].strip()

def medgemma_summarize_text(text):
    prompt = INSTRUCTIONS + text
    messages  = [{"role": "user", "content": prompt}]
    formatted = tokenizer_mg.apply_chat_template(
        messages, tokenize=False, add_generation_prompt=True
    )
    with warnings.catch_warnings():
        warnings.simplefilter("ignore")
        output = pipe_mg(formatted, return_full_text=False)
    return output[0]["generated_text"].strip()

print("Running MedGemma on all drugs (do not interrupt)...")
mg_results = []
mg_times   = []

num_texts = len(master_drug_data) * 2

for i, data in enumerate(master_drug_data):
    if i > 0 and i % 5 == 0:
        gc.collect()
        torch.cuda.empty_cache()

    for j, text in enumerate([data["section7"], data["section12"]]):
        if text is None:
          continue
        start = time.time()
        output = medgemma_summarize_text(text)
        elapsed = round(time.time() - start, 2)
        mg_results.append(output)
        mg_times.append(elapsed)
        master_drug_data[i][f"MedGemma_Summary_{j}"] = output
        master_drug_data[i][f"MedGemma_Time_{j}"] = elapsed

        if (i + 1 + j) % 5 == 0 or (i + 1 + j) == num_texts:
            free_gb = torch.cuda.mem_get_info()[0]/1e9
            print(f"  Progress: {i+1+j}/{num_texts} done — GPU free: {free_gb:.2f} GB")

mg_avg_time = round(sum(mg_times)/len(mg_times),2)

print(f"\n✓ MedGemma done. Avg time: {mg_avg_time}s")

Both `max_new_tokens` (=150) and `max_length`(=20) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


Running MedGemma on all drugs (do not interrupt)...


Both `max_new_tokens` (=150) and `max_length`(=20) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)
Both `max_new_tokens` (=150) and `max_length`(=20) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)
Both `max_new_tokens` (=150) and `max_length`(=20) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)
Both `max_new_tokens` (=150) and `max_length`(=20) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)
Both

  Progress: 5/146 done — GPU free: 32.08 GB


Both `max_new_tokens` (=150) and `max_length`(=20) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


...

  Progress: 70/146 done — GPU free: 32.00 GB


Both `max_new_tokens` (=150) and `max_length`(=20) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


  Progress: 70/146 done — GPU free: 32.00 GB


Both `max_new_tokens` (=150) and `max_length`(=20) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)
Both `max_new_tokens` (=150) and `max_length`(=20) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)
Both `max_new_tokens` (=150) and `max_length`(=20) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)
Both `max_new_tokens` (=150) and `max_length`(=20) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)
Both


✓ MedGemma done. Avg time: 10.11s


In [13]:
mg_avg_time = round(sum(mg_times)/len(mg_times),2)

print(f"\n✓ MedGemma done. Avg time: {mg_avg_time}s")


✓ MedGemma done. Avg time: 10.11s


#### Save Results to CSV

In [15]:
import csv

keys = master_drug_data[0].keys()

with open('drug_data_MedGemma_results.csv', 'w', newline='') as output_file:
    dict_writer = csv.DictWriter(output_file, keys)
    dict_writer.writeheader()
    dict_writer.writerows(master_drug_data)

print("Data saved to 'drug_data_MedGemma_results.csv'")

Data saved to 'drug_data_MedGemma_results.csv'


### Gemma 4 Inference

#### Remove MedGemma From Memory

In [16]:
import gc
import torch

try:
    del model_mg
    del pipe_mg
    gc.collect()
    torch.cuda.empty_cache()
    print(f"GPU cleared. Free: {torch.cuda.mem_get_info()[0]/1e9:.2f} GB")
except:
    print("No MedGemma in memory, continuing...")

GPU cleared. Free: 41.83 GB


#### Load Gemma 4

In [26]:
from transformers import AutoTokenizer, AutoModelForCausalLM, pipeline
import torch

MODEL_ID = "google/gemma-4-e4b-it"

# Load model and tokenizer once at module level to avoid reloading on each call
print(f"Loading model '{MODEL_ID}'...")
g4_tokenizer = AutoTokenizer.from_pretrained(MODEL_ID)
g4_model = AutoModelForCausalLM.from_pretrained(
    MODEL_ID,
    device_map="auto",
    dtype=torch.bfloat16,
)
print("Model loaded successfully.\n")

Loading model 'google/gemma-4-e4b-it'...


`torch_dtype` is deprecated! Use `dtype` instead!


model.safetensors:   0%|          | 0.00/16.0G [00:00<?, ?B/s]

Loading weights:   0%|          | 0/2076 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/208 [00:00<?, ?B/s]

Model loaded successfully.



#### Gemma 4 Summarize Function

In [29]:
def gemma4_summarize_text(text):
    messages = [
        {
            "role": "user",
            "content": INSTRUCTIONS + text,
        }
    ]

    # Apply the chat template and tokenize
    inputs = g4_tokenizer.apply_chat_template(
        messages,
        return_tensors="pt",
        return_dict=True,
        add_generation_prompt=True,
    ).to(g4_model.device)

    input_len = inputs["input_ids"].shape[-1]

    with torch.inference_mode():
        outputs = g4_model.generate(
            **inputs,
            max_new_tokens=250,
            do_sample=False,
        )

    summary = g4_tokenizer.decode(
        outputs[0][input_len:],
        skip_special_tokens=True,
    ).strip()

    return summary

#### Iterate Over Drug Data

In [30]:
gemma4_summarize_text(
"""
Without, the night was cold and wet, but in the small parlour of Laburnum Villa the blinds were drawn and the fire burned brightly. Father and son were at chess; the former, who possessed ideas about the game involving radical chances, putting his king into such sharp and unnecessary perils that it even provoked comment from the white-haired old lady knitting placidly by the fire.

"Hark at the wind," said Mr. White, who, having seen a fatal mistake after it was too late, was amiably desirous of preventing his son from seeing it.

"I'm listening," said the latter, grimly surveying the board as he stretched out his hand. "Check."

"I should hardly think he'll come to-night," said his father, with his hand poised over the board.

"Mate," replied the son.

"That's the worst of living so far out," bawled Mr. White, with sudden and unlooked-for violence. "Of all the beastly, slushy, out-of-the-way places to live in, this is the worst. Path's a bog, and the road's a torrent. I don't know what people are thinking about. I suppose because only two houses in the road are let, they think it doesn't matter."

"Never mind, dear," said his wife soothingly; "perhaps you'll win the next one."

Mr. White looked up sharply, just in time to intercept a knowing glance between mother and son. The words died away on his lips, and he hid a guilty grin in his thin grey beard.

"There he is," said Herbert White, as the gate banged too loudly and heavy footsteps came toward the door.

The old man rose with hospitable haste, and opening the door, was heard condoling with the new arrival. The new arrival also condoled with himself, so that Mrs. White said, "Tut, tut!" and coughed gently as her husband entered the room followed by a tall, burly man, beady of eye and rubicund of visage.

"Sergeant-Major Morris," he said, introducing him.

The Sergeant-Major took hands, and taking the proffered seat by the fire, watched contentedly as his host got out whisky and tumblers and stood a small copper kettle on the fire.

""")

The following generation flags are not valid and may be ignored: ['top_p', 'top_k']. Set `TRANSFORMERS_VERBOSITY=info` for more details.


'Indications: Information not found in source.\nDosing/Administration: Information not found in source.\nWarnings: Information not found in source.\nSide Effects: Information not found in source.\nDrug Interactions: Information not found in source.'

In [31]:
from transformers import AutoTokenizer, AutoModelForCausalLM, pipeline
import gc
import torch
import time
import warnings
import textwrap

print("Running Gemma 4 on all drugs (do not interrupt)...")
g4_results = []
g4_times   = []

num_texts = len(master_drug_data) * 2

for i, data in enumerate(master_drug_data):
    if i > 0 and i % 5 == 0:
        gc.collect()
        torch.cuda.empty_cache()

    for j, text in enumerate([data["section7"], data["section12"]]):
        if text is None:
          continue
        start = time.time()
        output = gemma4_summarize_text(text)
        elapsed = round(time.time() - start, 2)
        g4_results.append(output)
        g4_times.append(elapsed)
        master_drug_data[i][f"Gemma4_Summary_{j}"] = output
        master_drug_data[i][f"Gemma4_Time_{j}"] = elapsed

        if (i + 1 + j) % 5 == 0 or (i + 1 + j) == num_texts:
            free_gb = torch.cuda.mem_get_info()[0]/1e9
            print(f"  Progress: {i+1+j}/{num_texts} done — GPU free: {free_gb:.2f} GB")

g4_avg_time = round(sum(g4_times)/len(g4_times),2)

print(f"\n✓ Gemma 4 done. Avg time: {g4_avg_time}s")

Running Gemma 4 on all drugs (do not interrupt)...
  Progress: 5/146 done — GPU free: 7.76 GB
  Progress: 5/146 done — GPU free: 7.76 GB
  Progress: 10/146 done — GPU free: 8.51 GB
  Progress: 10/146 done — GPU free: 8.51 GB
  Progress: 15/146 done — GPU free: 8.91 GB
  Progress: 15/146 done — GPU free: 8.91 GB
  Progress: 20/146 done — GPU free: 9.12 GB
  Progress: 20/146 done — GPU free: 9.12 GB
  Progress: 25/146 done — GPU free: 6.06 GB
  Progress: 25/146 done — GPU free: 6.06 GB
  Progress: 30/146 done — GPU free: 7.02 GB
  Progress: 30/146 done — GPU free: 7.02 GB
  Progress: 35/146 done — GPU free: 7.69 GB
  Progress: 35/146 done — GPU free: 7.69 GB
  Progress: 40/146 done — GPU free: 9.13 GB
  Progress: 40/146 done — GPU free: 9.13 GB
  Progress: 45/146 done — GPU free: 9.59 GB
  Progress: 45/146 done — GPU free: 9.59 GB
  Progress: 50/146 done — GPU free: 8.88 GB
  Progress: 50/146 done — GPU free: 8.88 GB
  Progress: 55/146 done — GPU free: 8.30 GB
  Progress: 55/146 done — G

#### Save Results to CSV

In [32]:
import csv

keys = master_drug_data[0].keys()

with open('drug_data_Gemma4_results.csv', 'w', newline='') as output_file:
    dict_writer = csv.DictWriter(output_file, keys)
    dict_writer.writeheader()
    dict_writer.writerows(master_drug_data)

print("Data saved to 'drug_data_Gemma4_results.csv'")

Data saved to 'drug_data_Gemma4_results.csv'


### Results

#### Setup for BERTScores

In [33]:
import logging
import os

from transformers import AutoTokenizer, AutoModel
from bert_score import BERTScorer
import torch

logging.getLogger("transformers").setLevel(logging.ERROR)
os.environ["TOKENIZERS_PARALLELISM"] = "false"

def wrap_output(text, width=80, indent="  "):
    return "\n".join(textwrap.wrap(
        text.strip(), width=width,
        initial_indent=indent, subsequent_indent=indent
    ))

def get_bertscore_batch(hypotheses, references):
    MED_MODEL = "microsoft/BiomedNLP-PubMedBERT-base-uncased-abstract-fulltext"

    tokenizer = AutoTokenizer.from_pretrained(MED_MODEL)

    if not hasattr(tokenizer, "build_inputs_with_special_tokens"):
        def build_inputs_with_special_tokens(token_ids_0, token_ids_1=None):
            sep = [tokenizer.sep_token_id]
            cls = [tokenizer.cls_token_id]
            if token_ids_1 is None:
                return cls + token_ids_0 + sep
            return cls + token_ids_0 + sep + token_ids_1 + sep
        tokenizer.build_inputs_with_special_tokens = build_inputs_with_special_tokens

    if tokenizer.model_max_length > 1_000_000:
        tokenizer.model_max_length = 512

    scorer = BERTScorer(
        model_type=MED_MODEL,
        num_layers=9,
        batch_size=16,
        lang="en",
        use_fast_tokenizer=True
    )

    scorer._tokenizer = tokenizer

    with torch.no_grad():
        P, R, F1 = scorer.score(hypotheses, references)

    return [round(f.item(), 4) for f in F1]

references = []
for data in master_drug_data:
    for text in [data["section7"], data["section12"]]:
        if text is None:
            continue
        references.append(text)

print(len(lt5_results), len(mg_results), len(g4_results), len(references))

146 146 146 146


#### Calculate BERTScores

In [34]:
print("Scoring all outputs with BERTScore...")
lt5_scores = get_bertscore_batch(lt5_results, references)
mg_scores  = get_bertscore_batch(mg_results,  references)
g4_scores  = get_bertscore_batch(g4_results,  references)
print("Done.\n")

Scoring all outputs with BERTScore...


config.json:   0%|          | 0.00/385 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/28.0 [00:00<?, ?B/s]

vocab.txt: 0.00B [00:00, ?B/s]

pytorch_model.bin:   0%|          | 0.00/440M [00:00<?, ?B/s]

Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

Done.



In [35]:
print(lt5_scores, mg_scores, g4_scores)

[0.8958, 0.9016, 0.8898, 0.823, 0.8885, 0.7943, 0.8781, 0.8718, 0.9087, 0.8049, 0.9024, 0.8488, 0.8096, 0.7953, 0.0, 0.9053, 0.8866, 0.797, 0.8722, 0.8525, 0.9103, 0.8097, 0.8749, 0.7857, 0.8731, 0.8805, 0.8856, 0.8744, 0.858, 0.7994, 0.9073, 0.8698, 0.8952, 0.7937, 0.8463, 0.8118, 0.9409, 0.8405, 0.8804, 0.8343, 0.9056, 0.8102, 0.8854, 0.8796, 0.8445, 0.8681, 0.8307, 0.8344, 0.884, 0.8658, 0.8669, 0.7396, 0.844, 0.8205, 0.87, 0.8562, 0.8688, 0.888, 0.8165, 0.8536, 0.8419, 0.7899, 0.8659, 0.7848, 0.8588, 0.8484, 0.9048, 0.8584, 0.8619, 0.889, 0.878, 0.7652, 0.9086, 0.8741, 0.8795, 0.8512, 0.0, 0.8454, 0.8697, 0.8532, 0.8554, 0.8461, 0.888, 0.7945, 0.8674, 0.8156, 0.8146, 0.9386, 0.8521, 0.7944, 0.8443, 0.85, 0.8942, 0.8096, 0.9083, 0.8878, 0.8307, 0.8624, 0.8514, 0.7703, 0.8669, 0.8614, 0.8367, 0.772, 0.8926, 0.7899, 0.8227, 0.8062, 0.8355, 0.8371, 0.91, 0.8498, 0.8731, 0.8944, 0.8695, 0.8174, 0.8883, 0.8647, 0.8881, 0.871, 0.8321, 0.8107, 0.838, 0.754, 0.9284, 0.8393, 0.9493, 0.8445, 

#### Add BERTScores to Data

In [42]:
def add_bert_scores(data_list, score_list, prefix=""):
    if len(data_list) * 2 != len(score_list):
        raise ValueError(
            f"Mismatched lengths: {len(data_list)} dicts require {len(data_list)*2} scores, "
            f"but got {len(score_list)} scores."
        )

    for i, data_dict in enumerate(data_list):
        base_idx = i * 2
        data_dict[f"{prefix}BERTScore_Sec7"] = score_list[base_idx]
        data_dict[f"{prefix}BERTScore_Sec12"] = score_list[base_idx + 1]

    return data_list

In [43]:
master_drug_data = add_bert_scores(master_drug_data, lt5_scores, "LongT5_")

In [44]:
master_drug_data = add_bert_scores(master_drug_data, mg_scores, "MedGemma_")

In [45]:
master_drug_data = add_bert_scores(master_drug_data, g4_scores, "Gemma4_")

### Save results to CSV

In [46]:
import csv

keys = master_drug_data[0].keys()

with open('drug_data_BERTScore_results.csv', 'w', newline='') as output_file:
    dict_writer = csv.DictWriter(output_file, keys)
    dict_writer.writeheader()
    dict_writer.writerows(master_drug_data)

print("Data saved to 'drug_data_BERTScore_results.csv'")

Data saved to 'drug_data_BERTScore_results.csv'


#### Select Three Examples

In [47]:
import random

showcase_indices = []

for i in range(4):
    random_index = random.randint(0, len(master_drug_data) - 1)
    showcase_indices.append(random_index)

showcase_cases = [master_drug_data[i] for i in showcase_indices if i < len(master_drug_data)]

print(f"Showcase cases  : {len(showcase_cases) * 2}")
print(f"Total test cases: {len(master_drug_data) * 2}")
for c in showcase_cases:
    print(f"  [{c['index']}] {c['url'][:70]}")

Showcase cases  : 8
Total test cases: 146
  [96] https://dailymed.nlm.nih.gov/dailymed/fda/fdaDrugXsl.cfm?setid=536735f
  [32] https://dailymed.nlm.nih.gov/dailymed/fda/fdaDrugXsl.cfm?setid=696f9e8
  [17] https://www.novartis.us/sites/www.novartis.us/files/piqray.pdf
  [99] https://products.sanofi.us/ambien/ambien.pdf


#### Showcase Examples

In [48]:
print("="*70)
print("  SHOWCASE — 3 EXAMPLE SUMMARIES")
print("="*70)

for case in showcase_cases:
    idx = next(i for i, c in enumerate(master_drug_data) if c["url"] == case["url"])

    print(f"\n{'─'*70}")
    print(f"  DRUG {idx+1}")
    print(f"  Source: {case['url'][:70]}")
    print(f"{'─'*70}")

    print(f"\n  ORIGINAL TEXT (first 150 words):")
    print(wrap_output(" ".join(case["text"].split()[:150])))

    print(f"\n  LONG-T5 SUMMARY:")
    print(wrap_output(lt5_results[idx]))
    print(f"\n  ⏱ Time: {lt5_times[idx]}s  |  BERTScore F1: {lt5_scores[idx]}")

    print(f"\n  MEDGEMMA SUMMARY:")
    print(wrap_output(mg_results[idx]))
    print(f"\n  ⏱ Time: {mg_times[idx]}s  |  BERTScore F1: {mg_scores[idx]}")

    print(f"\n  GEMMA4 SUMMARY:")
    print(wrap_output(g4_results[idx]))
    print(f"\n  ⏱ Time: {g4_times[idx]}s  |  BERTScore F1: {g4_scores[idx]}")


  SHOWCASE — 3 EXAMPLE SUMMARIES

──────────────────────────────────────────────────────────────────────
  DRUG 70
  Source: https://dailymed.nlm.nih.gov/dailymed/fda/fdaDrugXsl.cfm?setid=536735f
──────────────────────────────────────────────────────────────────────

  ORIGINAL TEXT (first 150 words):


KeyError: 'text'

#### Overall Comparison

In [49]:
avg_lt5_score = round(sum(lt5_scores) / len(lt5_scores), 4)
avg_mg_score  = round(sum(mg_scores)  / len(mg_scores),  4)
avg_g4_score  = round(sum(g4_scores)  / len(g4_scores),  4)

avg_lt5_time  = round(sum(lt5_times)  / len(lt5_times),  2)
avg_mg_time   = round(sum(mg_times)   / len(mg_times),   2)
avg_g4_time   = round(sum(g4_times)   / len(g4_times),   2)

scores = {
    "Long-T5": avg_lt5_score,
    "MedGemma": avg_mg_score,
    "Gemma4": avg_g4_score
}

times = {
    "Long-T5": avg_lt5_time,
    "MedGemma": avg_mg_time,
    "Gemma4": avg_g4_time
}

winner = max(scores, key=scores.get)
faster = min(times, key=times.get)

print(f"\n{'='*75}")
print("  OVERALL RESULTS")
print(f"{'='*75}")
print(f"  {'Metric':<30} {'Long-T5':>10} {'MedGemma':>10} {'Gemma4':>10}")
print(f"  {'─'*70}")
print(f"  {'Avg BERTScore F1':<30} {avg_lt5_score:>10} {avg_mg_score:>10} {avg_g4_score:>10}")
print(f"  {'Avg Time per Summary (s)':<30} {avg_lt5_time:>10} {avg_mg_time:>10} {avg_g4_time:>10}")
print(f"  {'Total Drugs Evaluated':<30} {len(master_drug_data):>10}")
print(f"{'─'*75}")
print(f"  More Accurate Model : {winner} ({scores[winner]})")
print(f"  Faster Model        : {faster} ({times[faster]}s)")
print(f"{'='*75}")


  OVERALL RESULTS
  Metric                            Long-T5   MedGemma     Gemma4
  ──────────────────────────────────────────────────────────────────────
  Avg BERTScore F1                   0.8366     0.7903     0.7939
  Avg Time per Summary (s)             5.45      10.11      14.84
  Total Drugs Evaluated                  73
───────────────────────────────────────────────────────────────────────────
  More Accurate Model : Long-T5 (0.8366)
  Faster Model        : Long-T5 (5.45s)
